**Open-Source Model Run Script**
- plug in HuggingFace model link into *model_name*

In [ ]:
import os
import torch
import pandas as pd
from typing import List
from generate_prompts import create_prompt_list
from transformers import AutoTokenizer, AutoModelForCausalLM

# Check GPU
if not torch.cuda.is_available():
    raise SystemError("GPU not available.")

# Load the model, tokenizer
model_name = "mosaicml/mpt-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

# Generate model responses
def evaluate_prompts(prompts: List[str]) -> List[str]:
    responses = []

    for prompt in prompts:
        input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_length = input_ids.input_ids.shape[1]

        output = model.generate(
            **input_ids,
            max_new_tokens=500,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            return_dict_in_generate=True
        )

        generated_tokens = output.sequences[0, input_length:]
        decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        responses.append(decoded.strip())

    return responses

# Load constructed tests
data_df = pd.read_csv("tests.csv")
prompts = create_prompt_list(data_df)
responses = evaluate_prompts(prompts)

# Save responses to .txt file
with open("model-responses.txt", "w") as f:
    for idx, (prompt, response) in enumerate(zip(prompts, responses), start=1):
        f.write(f"Prompt {idx}:\n{prompt}\n")
        f.write(f"Hugging Face Response:\n{response}\n")
        f.write('-' * 40 + "\n")

print("Responses have been saved to 'model-responses.txt'")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for o

✅ Responses have been saved to 'incite-base-7b_responses.txt'.


In [ ]:
!git clone https://github.com/huggingface/transformers.git
%cd transformers
!pip install -e .

fatal: destination path 'transformers' already exists and is not an empty directory.
/content/transformers
Obtaining file:///content/transformers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for transformers (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.52.0.dev0-0.editable-py3-none-any.whl size=15239 sha256=b86f40b556252207dae6cb0e4ef2bf9b780fbd47036bcec8448e47535ac36d33
  Stored in directory: /tmp/pip-ephem-wheel-cache-4568qvln/wheels/9f/62/72/77fdff469e8308ad837268261590df9cabff9926cc4ab177c0
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.0.dev0
    Uninstalling transformers-4.52.0.dev0:
      Successfully uninstalled transformers-4.52.0.dev0


- upload *generate_prompts.py* and *tests.csv* before running prompt script

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving generate_prompts.py to generate_prompts.py
Saving tests.csv to tests.csv


- run this to save responses locally:

In [ ]:
from google.colab import files
files.download("model-responses.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

* prep environment accordingly:

In [ ]:
!pip install transformers accelerate bitsandbytes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 9.2 MB/s eta 0:00:00
